# Experiment 8 - Detecting Three Straight Lines with the Hough Transform

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
H, W = 300, 400
canvas = np.zeros((H, W), dtype=np.uint8)
cv2.line(canvas, (20, 260), (380, 40), 255, 2)
cv2.line(canvas, (60, 30), (90, 280), 255, 2)
cv2.line(canvas, (10, 150), (390, 190), 255, 2)

rng = np.random.default_rng(11)
noise = rng.random(canvas.shape) < 0.01
scene = canvas.copy()
scene[noise] = 255

edges = cv2.Canny(scene, 50, 150)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(scene, cmap="gray")
axes[0].set_title("three lines + noise")
axes[1].imshow(edges, cmap="gray")
axes[1].set_title("canny edges")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
def hough(edge_img, theta_step=1.0):
    thetas = np.deg2rad(np.arange(-90, 90, theta_step))
    diag = int(np.ceil(np.hypot(*edge_img.shape)))
    rhos = np.arange(-diag, diag + 1)
    acc = np.zeros((len(rhos), len(thetas)), dtype=np.int32)
    ys, xs = np.nonzero(edge_img)
    cos_t, sin_t = np.cos(thetas), np.sin(thetas)
    r = np.round(np.outer(xs, cos_t) + np.outer(ys, sin_t)).astype(int) + diag
    cols = np.broadcast_to(np.arange(len(thetas)), r.shape)
    np.add.at(acc, (r.ravel(), cols.ravel()), 1)
    return acc, rhos, thetas

acc, rhos, thetas = hough(edges)

plt.figure(figsize=(8, 6))
plt.imshow(acc, cmap="inferno", aspect="auto",
           extent=[np.rad2deg(thetas[0]), np.rad2deg(thetas[-1]), rhos[-1], rhos[0]])
plt.xlabel("theta (deg)")
plt.ylabel("rho (px)")
plt.title("hough accumulator")
plt.colorbar()
plt.show()